In [1]:
from models import TransferSegmentation, save_model, load_model, UNet
import torch 
import torch.nn as nn
from torchvision.transforms import v2
from utils import load_data, DATASET_PATH
import torch.utils.tensorboard as tb 
import numpy as np 

def calculate_iou(predicted_mask, target_mask, epsilon=1e-6):
    predicted_mask = torch.sigmoid(predicted_mask) # probabilidades
    predicted_mask = (predicted_mask > 0.5).float() # matriz true/false

    predicted_mask = predicted_mask.view(-1).bool() # aplanar la matriz
    target_mask = target_mask.view(-1).bool() # aplazar la matriz

    intersection = (predicted_mask & target_mask).float().sum() # coincidencias predicted_mask == target_mask == 1
    union = (predicted_mask | target_mask).float().sum() # suma de trues si predicted_mask == 1 o target_mask == 1

    iou = (intersection + epsilon)/(union + epsilon)

    return iou.item()

In [2]:
import datetime 

train_logger = None 
log_dir = f'semantic_segmentation/runs/{datetime.datetime.now().strftime("%Y%m%d-%H%M%s")}'

if log_dir:
    from os import path 
    train_logger = tb.SummaryWriter(path.join(log_dir), 'train', flush_secs=1)
    

In [ ]:
model = UNet(n_classes=1)

transform = v2.Compose([
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),
    v2.Resize((224, 224))
])

# transform = model.weights.transforms() # transformaciones para entrenar el backbone

train_loader = load_data(dataset_path=DATASET_PATH, transform=transform, batch_size=64)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model.to(device)

criterion = nn.BCEWithLogitsLoss() # recibe directamente los logits (la salida bruta del modelo)
optimizer = torch.optim.Adagrad(model.parameters(), lr=0.001)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max')

In [4]:
# Training Loop
best_iou = 0
num_epochs = 100
i = 0
patience = 0
max_patience = 15

for epoch in range(num_epochs):
    running_loss = 0
    epoch_mean_iou = []

    for (inputs, targets) in train_loader:
        inputs = inputs.to(device)
        targets = targets.to(device)

        outputs = model(inputs)
        loss = criterion(outputs, targets)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step() 

        batch_iou = calculate_iou(outputs, targets)
        epoch_mean_iou.append(batch_iou)

        running_loss += loss.item()

        if log_dir is not None:
            if i==0:
                train_logger.add_graph(model, inputs) # el diagrama del modelo

            train_logger.add_scalar('batch_loss', loss.item()/len(outputs), i)
            train_logger.add_scalar('batch_iou', batch_iou, i)
            train_logger.add_images('imagenes', inputs[:6], i)
            train_logger.add_images('targets', targets[:6], i)
            train_logger.add_images('preds', torch.sigmoid(outputs[:6])>0.5, i)
            
            i += 1
    
    # Estadisticas de la epoca
    epoch_mean_iou = np.mean(epoch_mean_iou) * 100
    epoch_loss = running_loss / len(train_loader)

    scheduler.step(epoch_mean_iou)

    print(f"Epoch {epoch+1}/{num_epochs} Loss: {epoch_loss:.2f} Mean IoU: {epoch_mean_iou:.2f}%")
    if epoch_mean_iou > best_iou:
        best_iou = epoch_mean_iou
        save_model(model, f'{model._get_name()}.th')
        patience = 0
    else:
        patience += 1

    if patience == max_patience:
        break

Epoch 1/100 Loss: 0.67 Mean IoU: 0.11%
Epoch 2/100 Loss: 0.61 Mean IoU: 4.66%
Epoch 3/100 Loss: 0.56 Mean IoU: 11.94%
Epoch 4/100 Loss: 0.53 Mean IoU: 19.62%
Epoch 5/100 Loss: 0.51 Mean IoU: 28.62%
Epoch 6/100 Loss: 0.50 Mean IoU: 35.93%
Epoch 7/100 Loss: 0.50 Mean IoU: 40.71%
Epoch 8/100 Loss: 0.48 Mean IoU: 47.38%
Epoch 9/100 Loss: 0.47 Mean IoU: 52.09%
Epoch 10/100 Loss: 0.46 Mean IoU: 55.18%
Epoch 11/100 Loss: 0.45 Mean IoU: 57.12%
Epoch 12/100 Loss: 0.44 Mean IoU: 59.61%
Epoch 13/100 Loss: 0.43 Mean IoU: 61.75%
Epoch 14/100 Loss: 0.42 Mean IoU: 62.39%
Epoch 15/100 Loss: 0.40 Mean IoU: 64.26%
Epoch 16/100 Loss: 0.39 Mean IoU: 64.97%
Epoch 17/100 Loss: 0.37 Mean IoU: 66.23%
Epoch 18/100 Loss: 0.37 Mean IoU: 66.12%
Epoch 19/100 Loss: 0.36 Mean IoU: 67.02%
Epoch 20/100 Loss: 0.35 Mean IoU: 67.35%
Epoch 21/100 Loss: 0.35 Mean IoU: 67.56%
Epoch 22/100 Loss: 0.35 Mean IoU: 66.59%
Epoch 23/100 Loss: 0.34 Mean IoU: 68.32%
Epoch 24/100 Loss: 0.33 Mean IoU: 69.14%
Epoch 25/100 Loss: 0.33 Mea

KeyboardInterrupt: 